# 06 — zitadel-gql conformance

The jaen identity client against the zitadel-gql facade SDL: every
query/mutation the CMS uses must exist with the right args, and the
legacy REST usergrant path must be gone from the auth context.

Live checks run only when `JAEN_ZITADEL_GQL_URL` answers. No public
zitadel-gql deployment is reachable from this machine today — the
walther-estate instances (`ghcr.io/netsnek/zitadel-gql`) sit behind
the university network, and the netcup Zitadels are stock v4 — so
the SDL at `JAEN_IAM_SDL` is the conformance source of record.


In [ ]:
import jaen_testkit as k
k.start_run('06-zitadel-gql')
print(k.CONFIG['repo_root'])

In [ ]:
import re

SDL = k.read_text(k.CONFIG['iam_sdl'], '')
CLIENT = k.read_text(k.repo_path('packages/jaen/src/clients/zitadel-gql/index.ts'), '')

def sdl_fields(type_name):
    m = re.search(r'type %s \{(.*?)\}' % type_name, SDL, re.S)
    if not m:
        return {}
    fields = {}
    for line in m.group(1).splitlines():
        fm = re.match(r'\s*(\w+)\s*\(([^)]*)\)\s*:\s*(.+)', line) or \
             re.match(r'\s*(\w+)\s*:\s*(.+)', line)
        if fm:
            fields[fm.group(1)] = line.strip()
    return fields

queries = sdl_fields('Query')
mutations = sdl_fields('Mutation')

with k.section('SDL sanity'):
    with k.check('facade SDL has the expected roots') as c:
        if not SDL:
            c.skip('no SDL available')
        for f in ('currentUser', 'users', 'user', 'projectRoles'):
            c.expect_true(f in queries, f)
        for f in ('createUser', 'updateUser', 'setUserPassword',
                  'createAuthorization', 'deleteAuthorization'):
            c.expect_true(f in mutations, f)


In [ ]:
with k.section('client conformance'):
    used_ops = set(re.findall(r'\b(?:query|mutation) Jaen\w+', CLIENT))
    field_calls = set(re.findall(r'^\s{8,10}(\w+)\(args:', CLIENT, re.M))

    with k.check('every client operation field exists in the SDL') as c:
        if not SDL:
            c.skip('no SDL')
        known = set(queries) | set(mutations)
        unknown = sorted(f for f in field_calls if f not in known)
        c.note('%d ops used: %s' % (len(field_calls), sorted(field_calls)[:8]))
        c.expect_equal(unknown, [], 'no unknown fields')

    with k.check('client uses inline fragments for IUserNode') as c:
        c.expect_contains(CLIENT, '... on HumanUser')
        c.expect_contains(CLIENT, '... on MachineUser')

    with k.check('profile update maps to givenName/familyName') as c:
        auth_user = k.read_text(k.repo_path('packages/jaen/src/contexts/auth-user.tsx'), '')
        c.expect_contains(auth_user, 'givenName')
        c.expect_contains(auth_user, 'familyName')


In [ ]:
with k.section('legacy REST removal'):
    auth = k.read_text(k.repo_path('packages/jaen/src/contexts/auth.tsx'), '')
    with k.check('auth.tsx no longer calls /auth/v1/usergrants') as c:
        c.expect_not_contains(auth, '/auth/v1/usergrants')
    with k.check('auth.tsx keeps the token-claim fallback') as c:
        c.expect_contains(auth, 'urn:zitadel:iam:org:project:roles')
    with k.check('the renamed global is used consistently') as c:
        c.expect_not_contains(auth, '__JAEN_ZITADEL__.')
        c.expect_contains(auth, '__JAEN_ZITADEL_GQL__')


In [ ]:
with k.section('live facade (optional)'):
    with k.check('zitadel-gql endpoint answers') as c:
        url = k.CONFIG['zitadel_gql_url']
        if not url:
            c.skip('JAEN_ZITADEL_GQL_URL not set')
        r = k.graphql(url, '{ __typename }')
        if r.skipped or r.error:
            c.skip('endpoint not reachable')
        c.expect_true(r.status in (200, 401, 403),
                      'HTTP %s (auth-gated is fine)' % r.status)


In [ ]:
k.summary()
k.save_results('results-06-zitadel-gql.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'